# Extreme Feature Engineering Search (Competition Metric Optimized)

This notebook performs an automatic feature engineering search using `TabularAML`.
It has been customized to:
1.  **Optimize the OFFICIAL Exact Competition Metric**: Correct linear weighting, derived thresholds, and survey-aggregation.
2.  **Respect Data Independence**: Uses 3-Fold Group Cross-Validation on `survey_id`.
3.  **Use Extreme Search Settings**: 100 generations, 500 children, high exploration.

In [1]:
import pandas as pd
import numpy as np
import joblib
from tabularaml.generate.features import FeatureGenerator
from tabularaml.eval.scorers import Scorer
from sklearn.model_selection import GroupKFold

Added numpy() method to pandas Series


c:\ml_env\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


## 1. Load Data

In [2]:
# Load datasets
df_train_features = pd.read_csv('data/train_hh_features.csv')
df_train_gt = pd.read_csv('data/train_hh_gt.csv')

# Merge identifiers and target
# We also need 'survey_id' for Group K-Fold and for the Custom Metric
train_df = pd.merge(df_train_features, 
                    df_train_gt[['hhid', 'cons_ppp17', 'survey_id']], 
                    on='hhid', how='inner')

# Drop duplicate survey_id if exists/renaming
if 'survey_id_x' in train_df.columns:
    train_df = train_df.rename(columns={'survey_id_x': 'survey_id'})
    train_df.drop(columns=['survey_id_y'], inplace=True, errors='ignore')

# GLOBAL LOOKUP for Survey IDs (used by the Metric Scorer)
# We rely on the index of y_true matching this lookup table during scoring
SURVEY_ID_LOOKUP = train_df['survey_id'].copy()

print(f"Dataset shape: {train_df.shape}")
print(f"Unique Surveys: {train_df['survey_id'].unique()}")
train_df.head()

Dataset shape: (104234, 89)
Unique Surveys: [100000 200000 300000]


,hhid,com,weight,strata,utl_exp_ppp17,male,hsize,num_children5,num_children10,num_children18,...,consumed4300,consumed4400,consumed4500,consumed4600,consumed4700,consumed4800,consumed4900,consumed5000,survey_id,cons_ppp17
0,100001,1,75,4,594.80627,Female,1,0,0,0,...,No,No,No,Yes,Yes,Yes,Yes,No,100000,25.258402
1,100002,1,150,4,1676.27230,Female,2,0,0,0,...,No,No,No,No,Yes,Yes,No,No,100000,16.996706
2,100003,1,375,4,506.93719,Male,5,0,0,2,...,Yes,No,Yes,Yes,Yes,Yes,No,Yes,100000,13.671848
3,100004,1,375,4,824.61786,Male,5,0,0,1,...,Yes,No,No,No,Yes,Yes,No,No,100000,7.189475
4,100005,1,525,4,351.47644,Male,7,1,0,0,...,No,No,Yes,No,Yes,Yes,Yes,No,100000,12.308855


## 2. Define Exact Competition Metric

Implemented 1:1 with official rules:
- Weights: `w_t = 1 - |0.4 - p_t|` (Linear)
- Divide by sum of weights
- Per-survey Aggregation
- Thresholds derived from Survey 300000

In [10]:
import numpy as np

class CompetitionMetricExact:
    def __init__(self, eps: float = 1e-8):
        self.eps = float(eps)
        self.percentiles = np.arange(0.05, 1.0, 0.05)
        self.weights = 1.0 - np.abs(0.4 - self.percentiles)
        self.sum_w = float(np.sum(self.weights))
        self.thresholds = None

    def fit_thresholds_from_training(self, y_train_consumption, train_survey_ids, threshold_survey_id=300000):
        y_train_consumption = np.asarray(y_train_consumption, dtype=float)
        train_survey_ids = np.asarray(train_survey_ids)
        mask = (train_survey_ids == threshold_survey_id)
        vals = y_train_consumption[mask]
        if vals.size == 0:
            raise ValueError(f"No training rows found for survey_id={threshold_survey_id}.")
        self.thresholds = np.quantile(vals, self.percentiles)
        return self.thresholds

    def score(self, y_true, y_pred, survey_ids):
        if self.thresholds is None:
            raise ValueError("Thresholds not set.")

        yt = np.asarray(y_true, dtype=float)
        yp = np.asarray(y_pred, dtype=float)
        sid = np.asarray(survey_ids)

        n = yt.size
        if n == 0:
            return float("nan")  # consistent "nothing to score" behavior (your loop would divide by S=0)
        if yp.size != n or sid.size != n:
            raise ValueError("y_true, y_pred, survey_ids must have the same length.")

        # Sort by survey id so each group is contiguous
        order = np.argsort(sid, kind="mergesort")  # stable sort (not required, but nice)
        sid_s = sid[order]
        yt_s = yt[order]
        yp_s = yp[order]

        # Group boundaries
        change = np.empty(n, dtype=bool)
        change[0] = True
        change[1:] = sid_s[1:] != sid_s[:-1]
        starts = np.flatnonzero(change)                # start index of each group
        counts = np.diff(np.append(starts, n))         # group sizes
        S = starts.size                                # number of surveys

        eps = self.eps

        # --- Consumption MAPE per survey: mean(|yt-yp|/max(|yt|,eps)) ---
        denom_cons = np.maximum(np.abs(yt_s), eps)
        cons_term = np.abs(yt_s - yp_s) / denom_cons
        cons_sum = np.add.reduceat(cons_term, starts)
        cons_mape = cons_sum / counts  # shape (S,)

        # --- Poverty rates per survey per threshold ---
        thr = np.asarray(self.thresholds, dtype=float)  # (T,)
        T = thr.size

        # (N,T) comparisons, then group-mean them
        # true_rates[s,t] = mean(yt in survey s is < thr[t])
        true_below = (yt_s[:, None] < thr[None, :])
        pred_below = (yp_s[:, None] < thr[None, :])

        true_cnt = np.add.reduceat(true_below, starts, axis=0)  # (S,T)
        pred_cnt = np.add.reduceat(pred_below, starts, axis=0)  # (S,T)

        true_rates = true_cnt / counts[:, None]
        pred_rates = pred_cnt / counts[:, None]

        denom_rates = np.maximum(np.abs(true_rates), eps)
        rate_mape = np.abs(pred_rates - true_rates) / denom_rates  # (S,T)

        w = self.weights.astype(float)  # (T,)
        pov_part = (90.0 / self.sum_w) * (rate_mape * w[None, :]).sum(axis=1)  # (S,)
        cons_part = 10.0 * cons_mape  # (S,)

        total_per_survey = pov_part + cons_part
        return float(total_per_survey.mean())


# Initialize
comp_metric = CompetitionMetricExact()
comp_metric.fit_thresholds_from_training(
    train_df['cons_ppp17'].values,
    train_df['survey_id'].values
)

# Wrapper to fetch Survey IDs from Global Lookup
def score_wrapper(y_true, y_pred):
    # Note: TabularAML usually passes y_true as a Series during evaluation if index is preserved
    if hasattr(y_true, 'index'):
        try:
            s_ids = SURVEY_ID_LOOKUP.loc[y_true.index].values
        except:
            # Fallback (e.g. if indices are reset), assuming single-group chunks
            s_ids = np.zeros(len(y_true))
    else:
        s_ids = np.zeros(len(y_true))
    return comp_metric.score(y_true, y_pred, s_ids)

competition_scorer = Scorer(
    name="competition_loss_exact", 
    scorer=score_wrapper, 
    greater_is_better=False,
    extra_params={}
)

## 3. Configure Extreme Feature Search

In [12]:
# Prepare X and y
target_col = 'cons_ppp17'

groups = train_df['survey_id'].values
X = train_df.drop(columns=[target_col, 'survey_id'])
y = train_df[target_col]

# Define 3-Fold Group CV
group_cv = GroupKFold(n_splits=3)
cv_splits = list(group_cv.split(X, y, groups=groups))

# Defines a wrapper class to make list of splits compatible with framework
class PredefinedSplits:
    def __init__(self, splits):
        self.splits = splits
    def split(self, X, y=None, groups=None):
        for split in self.splits:
            yield split
    def get_n_splits(self, X=None, y=None, groups=None):
        return len(self.splits)

custom_cv = PredefinedSplits(cv_splits)

# Initialize FeatureGenerator
feature_gen = FeatureGenerator(
    mode=None,
    task="regression",
    scorer=competition_scorer, # USES EXACT METRIC
    
    # Aggressive Search settings
    n_generations=100,
    n_parents=100,
    n_children=500,
    
    min_pct_gain=0.0001,
    ranking_method="multi_criteria",
    
    early_stopping_iter=20,
    early_stopping_child_eval=100,
    
    exploration_factor=0.5,
    adaptive=True,
    
    cv=custom_cv,
    use_gpu=True,
    time_budget=None, 
    log_file="logs/extreme_search_custom.log",
    save_path="models/feature_search_extreme.pkl"
)

## 4. Run Search

We use `.generate()` to start the evolutionary search process.

In [13]:
Gen 0: Train competition_loss_exact=12.89660, Val competition_loss_exact=13.27056


SyntaxError: invalid syntax (1244917325.py, line 1)

In [ ]:
print("Starting Extreme Feature Search (Competition Mode)...")
X_transformed, best_pipeline, best_generation, best_interactions = feature_gen.generate(X, y)
print("Search Complete.")

Starting Extreme Feature Search (Competition Mode)...
Starting regression on cuda - 104234 samples, 87 features
Params: gen=100, parents=100, children=500, limit=inf, time_budget=Nones.
Gen 0: Train competition_loss_exact=12.89660, Val competition_loss_exact=13.27056


In [ ]:
output_path = 'data/train_features_extreme_custom.csv'
X_transformed.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

joblib.dump(feature_gen, 'feature_generator_extreme_custom.pkl')
print("Saved generator.")